In [5]:
print("Hello world")

Hello world


### Installing all necessary packages

In [6]:
%pip install scikit-learn pyelftools

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install numpy scipy pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install scikit-learn imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install xgboost catboost lightgbm

### Importing libraries

In [ ]:
import os
import pandas as pd

from elftools.elf.elffile import ELFFile


import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate



### Extracting the features into csv

In [ ]:
# ==========================
# CONFIG
# ==========================

BENIGN_DIR = "data/ELF_data/ELF_Dataset/Benignware"
MALWARE_DIR = "data/ELF_data/ELF_Dataset/Malware"
OUTPUT_CSV = "data/elf_dataset.csv"


# ==========================
# Helper: Check ELF
# ==========================

def is_elf(path):
    try:
        with open(path, "rb") as f:
            return f.read(4) == b"\x7fELF"
    except Exception as e:
        print(f"Error: {e}\n")
        return False


In [ ]:
# ==========================
# Feature Extraction from files
# ==========================

def extract_features(file_path):

    features = {}

    with open(file_path, "rb") as f:
        elf = ELFFile(f)

        # ----------------------
        # Initialize All Features to 0
        # ----------------------

        symbol_types = ["STT_OBJECT", "STT_FUNC", "STT_NOTYPE"]
        symbol_bindings = ["STB_LOCAL", "STB_GLOBAL", "STB_WEAK"]

        for t in symbol_types:
            features[t] = 0
        for b in symbol_bindings:
            features[b] = 0

        for t in symbol_types:
            for b in symbol_bindings:
                features[f"{t}_{b}"] = 0

        dynamic_tags = [
            "DT_NEEDED", "DT_RELA", "DT_RELASZ", "DT_RELAENT",
            "DT_PLTREL", "DT_PLTRELSZ", "DT_JMPREL",
            "DT_STRTAB", "DT_STRSZ",
            "DT_SYMTAB", "DT_SYMENT",
            "DT_INIT", "DT_FINI",
            "DT_INIT_ARRAY", "DT_FINI_ARRAY",
            "DT_INIT_ARRAYSZ", "DT_FINI_ARRAYSZ",
            "DT_VERNEED", "DT_VERNEEDNUM",
            "DT_DEBUG", "DT_NULL"
        ]

        for tag in dynamic_tags:
            features[tag] = 0

        # ----------------------
        # SYMBOL TABLE (.dynsym)
        # ----------------------

        dynsym = elf.get_section_by_name(".dynsym")

        if dynsym:
            for symbol in dynsym.iter_symbols():

                sym_type = symbol["st_info"]["type"]
                sym_bind = symbol["st_info"]["bind"]

                if sym_type in symbol_types:
                    features[sym_type] += 1

                if sym_bind in symbol_bindings:
                    features[sym_bind] += 1

                if sym_type in symbol_types and sym_bind in symbol_bindings:
                    features[f"{sym_type}_{sym_bind}"] += 1

        # ----------------------
        # DYNAMIC SECTION
        # ----------------------

        dynamic = elf.get_section_by_name(".dynamic")

        if dynamic:
            for tag in dynamic.iter_tags():
                tag_name = tag.entry.d_tag
                if tag_name in dynamic_tags:
                    features[tag_name] += 1

    return features


# ==========================
# Process Directory
# ==========================

def process_directory(directory, label):

    dataset = []

    for root, _, files in os.walk(directory):
        for filename in files:

            path = os.path.join(root, filename)

            if not is_elf(path):
                continue

            try:
                features = extract_features(path)
                features["label"] = label
                dataset.append(features)

            except Exception as e:
                print("Skipping:", path, e)

    return dataset


# ==========================
# MAIN
# ==========================

def make_csv():
    data = []

    data.extend(process_directory(BENIGN_DIR, 0))
    data.extend(process_directory(MALWARE_DIR, 1))

    df = pd.DataFrame(data)
    df.fillna(0, inplace=True)

    df.to_csv(OUTPUT_CSV, index=False)

    print("Dataset saved as:", OUTPUT_CSV)
    print("Total samples:", len(df))
    print(df["label"].value_counts())


make_csv()

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/rnano'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/rrsync'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/rtstat'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/static-sh'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/sudoedit'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/vmrest'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/x86_64-linux-gnu-cpp'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/x86_64-linux-gnu-g++'

Error Not an ELF: [Errno 2] No such file or directory: 'data/ELF_data/ELF_Dataset/Benignware/rbash'

Error Not an ELF: [Errno 2] No such file or directo

In [26]:
df = pd.read_csv("data/elf_dataset.csv")
df.head()

,STT_OBJECT,STT_FUNC,STT_NOTYPE,STB_LOCAL,STB_GLOBAL,STB_WEAK,STT_OBJECT_STB_LOCAL,STT_OBJECT_STB_GLOBAL,STT_OBJECT_STB_WEAK,STT_FUNC_STB_LOCAL,...,DT_FINI,DT_INIT_ARRAY,DT_FINI_ARRAY,DT_INIT_ARRAYSZ,DT_FINI_ARRAYSZ,DT_VERNEED,DT_VERNEEDNUM,DT_DEBUG,DT_NULL,label
0,9,56,4,1,62,6,0,7,2,0,...,1,1,1,1,1,1,1,1,1,0
1,2,6,4,1,7,4,0,2,0,0,...,1,1,1,1,1,1,1,1,1,0
2,2,53,4,1,54,4,0,2,0,0,...,1,1,1,1,1,1,1,1,1,0
3,5,42,4,1,46,4,0,5,0,0,...,1,1,1,1,1,1,1,1,1,0
4,4,67,4,1,70,4,0,4,0,0,...,1,1,1,1,1,1,1,1,1,0


In [27]:
df.columns

Index(['STT_OBJECT', 'STT_FUNC', 'STT_NOTYPE', 'STB_LOCAL', 'STB_GLOBAL',
       'STB_WEAK', 'STT_OBJECT_STB_LOCAL', 'STT_OBJECT_STB_GLOBAL',
       'STT_OBJECT_STB_WEAK', 'STT_FUNC_STB_LOCAL', 'STT_FUNC_STB_GLOBAL',
       'STT_FUNC_STB_WEAK', 'STT_NOTYPE_STB_LOCAL', 'STT_NOTYPE_STB_GLOBAL',
       'STT_NOTYPE_STB_WEAK', 'DT_NEEDED', 'DT_RELA', 'DT_RELASZ',
       'DT_RELAENT', 'DT_PLTREL', 'DT_PLTRELSZ', 'DT_JMPREL', 'DT_STRTAB',
       'DT_STRSZ', 'DT_SYMTAB', 'DT_SYMENT', 'DT_INIT', 'DT_FINI',
       'DT_INIT_ARRAY', 'DT_FINI_ARRAY', 'DT_INIT_ARRAYSZ', 'DT_FINI_ARRAYSZ',
       'DT_VERNEED', 'DT_VERNEEDNUM', 'DT_DEBUG', 'DT_NULL', 'label'],
      dtype='object')

In [30]:
df.shape

(972, 37)

In [28]:
df.describe()

,STT_OBJECT,STT_FUNC,STT_NOTYPE,STB_LOCAL,STB_GLOBAL,STB_WEAK,STT_OBJECT_STB_LOCAL,STT_OBJECT_STB_GLOBAL,STT_OBJECT_STB_WEAK,STT_FUNC_STB_LOCAL,...,DT_FINI,DT_INIT_ARRAY,DT_FINI_ARRAY,DT_INIT_ARRAYSZ,DT_FINI_ARRAYSZ,DT_VERNEED,DT_VERNEEDNUM,DT_DEBUG,DT_NULL,label
count,972.000000,972.000000,972.000000,972.000000,972.000000,972.000000,972.0,972.000000,972.000000,972.0,...,972.000000,972.000000,972.000000,972.000000,972.000000,972.00000,972.00000,972.000000,972.000000,972.000000
mean,12.601852,112.902263,2.556584,0.525720,110.570988,16.973251,0.0,9.889918,2.709877,0.0,...,0.497942,0.493827,0.505144,0.493827,0.505144,0.49177,0.49177,0.496914,0.516461,0.514403
std,130.873886,928.791523,2.982939,0.541156,676.230925,419.005816,0.0,79.504511,56.272437,0.0,...,0.502308,0.500219,0.500231,0.500219,0.500231,0.50019,0.50019,0.500248,0.499986,0.500050
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000
50%,0.000000,15.500000,4.000000,1.000000,18.000000,3.500000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,1.000000,0.000000,1.000000,0.00000,0.00000,0.000000,1.000000,1.000000
75%,6.000000,73.250000,4.000000,1.000000,79.000000,5.000000,0.0,5.000000,0.000000,0.0,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000
max,3873.000000,27136.000000,40.000000,6.000000,17953.000000,13059.000000,0.0,2138.000000,1735.000000,0.0,...,2.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000


### Using ensemble models to classify
- XGBoost
- LightGBM
- CatBoost
- ExtraTrees

#### Training followed by cross-validation

In [ ]:
# -------------------------
# Load Data
# -------------------------

df = pd.read_csv("data/elf_dataset.csv")  # use full dataset now

X = df.drop("label", axis=1)
y = df["label"]

print("Shape:", X.shape)

# -------------------------
# CV Setup
# -------------------------

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score),
    "f1": make_scorer(f1_score)
}

# -------------------------
# Models
# -------------------------

models = {
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        verbose=0,
        random_seed=42
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42
    )
}

# -------------------------
# Evaluation
# -------------------------

for name, model in models.items():
    print(f"\n===== {name} =====")

    results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    for metric in results:
        if metric.startswith("test_"):
            print(
                metric.replace("test_", "").upper(),
                "Mean:", np.mean(results[metric]),
                "Std:", np.std(results[metric])
            )

Shape: (972, 36)

===== XGBoost =====


ACCURACY Mean: 0.9975289452815227 Std: 0.0036044264640834947
ROC_AUC Mean: 0.9989566853303472 Std: 0.0021384931877357614
RECALL Mean: 0.9952 Std: 0.006997142273814367
PRECISION Mean: 1.0 Std: 0.0
F1 Mean: 0.997581848637125 Std: 0.0035293973440764567

===== LightGBM =====
[LightGBM] [Info] Number of positive: 400, number of negative: 378
[LightGBM] [Info] Number of positive: 400, number of negative: 377
[LightGBM] [Info] Number of positive: 400, number of negative: 378
[LightGBM] [Info] Number of positive: 400, number of negative: 377
[LightGBM] [Info] Number of positive: 400, number of negative: 377
[LightGBM] [Info] Number of positive: 400, number of negative: 377
[LightGBM] [Info] Number of positive: 400, number of negative: 378
[LightGBM] [Info] Number of positive: 400, number of negative: 378
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.607373 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, y

### Reducing the number of features and evaluating models

In [42]:
# Load data
df = pd.read_csv("data/elf_dataset.csv")
X = df.drop("label", axis=1)
y = df["label"]

# Train ExtraTrees
model = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42
)

model.fit(X, y)

# Get importance
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 20 Features:\n")
print(importances.head(20))

# Save ranking
importances.to_csv("data/feature_importance_ranking.csv")


Top 20 Features:

DT_RELAENT              0.145261
DT_RELASZ               0.139144
DT_RELA                 0.136271
DT_VERNEEDNUM           0.126548
DT_VERNEED              0.110070
DT_DEBUG                0.065153
DT_INIT                 0.039071
DT_FINI_ARRAY           0.034112
DT_FINI_ARRAYSZ         0.032792
DT_STRTAB               0.026516
DT_FINI                 0.024094
STT_NOTYPE_STB_WEAK     0.020578
DT_SYMENT               0.019447
DT_STRSZ                0.015918
DT_INIT_ARRAY           0.014218
DT_NULL                 0.010666
DT_SYMTAB               0.007166
STT_NOTYPE_STB_LOCAL    0.007071
STT_NOTYPE              0.005237
DT_PLTREL               0.005098
dtype: float64


### Choosing top 10 features and evaluating

In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score
from sklearn.ensemble import ExtraTreesClassifier

# Load full dataset
df = pd.read_csv("data/elf_dataset.csv")

# Load importance ranking
importance = pd.read_csv("data/feature_importance_ranking.csv", index_col=0, header=None)
importance.columns = ["score"]
importance = importance.sort_values("score", ascending=False)

# Choose top K
K = 10  # try 15 and later 20
top_features = importance.head(K).index.tolist()

print("Selected Features:", top_features)

X = df[top_features]
y = df["label"]

# CV setup
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score),
    "f1": make_scorer(f1_score)
}

model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42
)

results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("\n===== Reduced Feature Results with extra trees =====\n")

for metric in results:
    if metric.startswith("test_"):
        print(
            metric.replace("test_", "").upper(),
            "Mean:", np.mean(results[metric]),
            "Std:", np.std(results[metric])
        )

Selected Features: ['DT_RELAENT', 'DT_RELASZ', 'DT_RELA', 'DT_VERNEEDNUM', 'DT_VERNEED', 'DT_DEBUG', 'DT_INIT', 'DT_FINI_ARRAY', 'DT_FINI_ARRAYSZ', 'DT_STRTAB']

===== Reduced Feature Results with extra trees =====

ACCURACY Mean: 0.9979413164155432 Std: 0.002912550471541852
ROC_AUC Mean: 0.9989957670772677 Std: 0.00200849930070432
RECALL Mean: 0.996 Std: 0.005656854249492385
PRECISION Mean: 1.0 Std: 0.0
F1 Mean: 0.9979879193949546 Std: 0.002848394612124387
